<span style="color:lightgreen">

# Notebook extra 3

En este notebook se experimenta con el fine-tuning de modelos e hiperparámetros para la transcripción con traducción de whisper en cascada.

En lugar de usar NLLB, usaremos el modelo `google/madlad400-3b-mt`

Covost2 pt-en

</span>

# Fine-tuning of the NLLB model for Europarl-ST (Spanish to English)

In this notebook, we are going to explain how to fine-tune the [NLLB model](https://huggingface.co/docs/transformers/model_doc/nllb) on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST). First, we need to run the experiment L4.1 on the development set to generate data to be used in the fine tuning process. Once the fine tuning of the NLLB is carried out, we will evaluate the NLLB model performance over the Europarl-ST test data where speech transcriptions were obtained also in the L4.1 experiment. Remember that the initial performance of the NLLB model on the test data evaluated in experiment L4.3 was 9.3 of BLEU and 57.9% COMET.

We load the automatic speech transcriptions and reference translations from the csv file generated in experiment L4.1 on the dev data. Moreover, we split the data into train (17%) and validation (test) (83%) sets for fine tuning

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

# Gneral configurations
CONFIG = Configuration(
    model_name="facebook/nllb-200-distilled-600M",
    max_tok_length = 275,
    batch_size = 2,
    # test_batch_size = 32,
    max_epoch = 10,

)

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/notebooks_pt_en
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app


In [2]:
# Suppress warnings and unnecessary output
import os
import warnings
import logging

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

# Set logging levels
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("lightning").setLevel(logging.ERROR)
logging.getLogger("comet_ml").setLevel(logging.ERROR)

In [ ]:
import torch
import numpy as np
from evaluate import load
from datasets import load_dataset
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments
from maikol_utils.print_utils import print_color

def train_model(CONFIG: Configuration):
    raw_datasets = load_dataset("csv", data_files=os.path.join(CONFIG.RESULTS_PATH, "Extra2.1-L4.1_ASR_Whisper_Baseline_Covost2_pt-en.csv"))

    raw_datasets = raw_datasets.select_columns(
        [CONFIG.input_col,"translation"]
    )
    raw_datasets = raw_datasets["train"].train_test_split(test_size=CONFIG.test_split, shuffle=False)
    print_color(raw_datasets, color="cyan")
    raw_datasets = raw_datasets.filter(lambda x: x[CONFIG.input_col] is not None and x["translation"] is not None)


    # from flores200_codes import flores_codes
    tokenizer = AutoTokenizer.from_pretrained(
        CONFIG.model_name, 
        padding=True, 
        pad_to_multiple_of=8, 
        src_lang=CONFIG.src_code, 
        tgt_lang=CONFIG.tgt_code, 
        truncation=True, 
        max_length=CONFIG.max_tok_length,
    )


    def preprocess_function(sample):
        model_inputs = tokenizer(
            sample[CONFIG.input_col], 
            text_target = sample["translation"],
            truncation=True,
            max_length=CONFIG.max_tok_length,
        )
        return model_inputs

    tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)


    # Load with low precision
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    # Pass the quantization_config to the from_pretrained method.
    model = AutoModelForSeq2SeqLM.from_pretrained(
        CONFIG.model_name,
        # text_target = sample["translation"],
        quantization_config=quantization_config
    )

    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False, gradient_checkpointing_kwargs={'use_reentrant':False})

    config = LoraConfig(
        task_type="SEQ_2_SEQ_LM",
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
    )

    # get model and print parameters
    lora_model = get_peft_model(model, config)
    lora_model.print_trainable_parameters()

    # Collator
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer, 
        model=lora_model, 
        pad_to_multiple_of=8
    )

    # ===========================================
    #               EVALUATION
    # ===========================================
    metric = load("sacrebleu")

    def postprocess_text(preds, labels):
        preds = [pred.strip() for pred in preds]
        labels = [[label.strip()] for label in labels]

        return preds, labels

    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

        # Replace negative ids in the labels as we can't decode them.
        #labels = np.where(labels < 0, labels, tokenizer.pad_token_id)
        for i in range(len(labels)):
            labels[i] = [tokenizer.pad_token_id if j<0 else j for j in labels[i]]
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        # Some simple post-processing
        decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

        result = metric.compute(predictions=decoded_preds, references=decoded_labels)
        result = {"bleu": result["score"]}

        prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
        result["gen_len"] = np.mean(prediction_lens)
        result = {k: round(v, 4) for k, v in result.items()}
        return result


    # ===========================================
    #               TRAINING
    # ===========================================
    args = Seq2SeqTrainingArguments(
        os.path.join(CONFIG.MODELS_PATH, CONFIG.fine_tune_model_name),
        eval_strategy = "no",
        learning_rate=1e-4,
        per_device_train_batch_size=CONFIG.batch_size,
        per_device_eval_batch_size=CONFIG.batch_size,
        weight_decay=0.01,
        save_total_limit=2,
        num_train_epochs=CONFIG.max_epoch,
        predict_with_generate=True,
    )

    from transformers import Seq2SeqTrainer

    trainer = Seq2SeqTrainer(
        lora_model,
        args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['test'],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    return tokenizer, lora_model

## Evaluation of the fine tuned NLLL model on the Europarl-ST test set

Let us first load the default inference parameters of NLLB: 

In [ ]:
from datasets import load_dataset
from transformers import GenerationConfig
from whisper.normalizers.basic import BasicTextNormalizer
from evaluate import load
from maikol_utils.print_utils import print_color

def evaluate_model(CONFIG: Configuration, tokenizer, lora_model):
    # Load generation config
    generation_config = GenerationConfig.from_pretrained(CONFIG.model_name)
    # print_color(generation_config, color="italic")

    # Load evaluation dataset
    raw_datasets = load_dataset("csv", data_files=os.path.join(CONFIG.RESULTS_PATH, "Extra2.1-L4.1_ASR_Whisper_Baseline_Covost2_pt-en.csv"))
    # print_color(raw_datasets, color="italic")
    raw_datasets = raw_datasets.filter(lambda x: x[CONFIG.input_col] is not None and x["translation"] is not None)


    # Each sample pair is preprocessed according to the training needs of the model that has been finetuned:
    def preprocess_function(sample):
        model_inputs = tokenizer(
            sample[CONFIG.input_col], 
            text_target = sample["translation"],
            truncation=True,
            # max_length=CONFIG.max_tok_length,
        )
        return model_inputs

    tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)
    batch_tokenized_test = tokenized_datasets['train'].batch(CONFIG.test_batch_size)


    # Generate translations
    number_of_batches = len(batch_tokenized_test[CONFIG.input_col])
    output_sequences = []
    for i in range(number_of_batches):
        inputs = tokenizer(batch_tokenized_test[CONFIG.input_col][i], max_length=CONFIG.max_tok_length, truncation=False, return_tensors="pt", padding=True)
        output_batch = lora_model.generate(generation_config=generation_config, input_ids=inputs["input_ids"].cuda(), attention_mask=inputs["attention_mask"].cuda(), forced_bos_token_id=tokenizer.convert_tokens_to_ids(CONFIG.tgt_code), max_length = CONFIG.max_tok_length, num_beams=1, do_sample=False,)
        output_sequences.extend(output_batch.cpu())


    # Recover decoded
    decoded_preds = tokenizer.batch_decode(output_sequences, skip_special_tokens=True)
    references = tokenizer.batch_decode(tokenized_datasets["train"]["labels"], skip_special_tokens=True)


    # =========== METRICS =============
    normalizer = BasicTextNormalizer()

    decoded_preds_clean = [normalizer(text) for text in decoded_preds] if CONFIG.normalize else decoded_preds
    references_clean = [normalizer(text) for text in references] if CONFIG.normalize else references
    metric = load("sacrebleu")
    result = metric.compute(predictions=decoded_preds_clean, references=references_clean)
    print_color(f'BLEU score: {result["score"]:.1f}', color="green")

    comet_metric = load('comet')
    comet_score = comet_metric.compute(predictions=decoded_preds_clean, references=references_clean, sources=raw_datasets["train"][CONFIG.input_col])
    print_color(f"COMET: {comet_score['mean_score']:.2%}", color="green")

# Experiments

In [5]:
configurations = [
    {"input_col": "hypothesis", "test_split": 0.17},
    {"input_col": "hypothesis", "test_split": 0.30},
    {"input_col": "hypothesis", "test_split": 0.50},
    {"input_col": "reference", "test_split": 0.17},
    {"input_col": "reference", "test_split": 0.30},
    {"input_col": "reference", "test_split": 0.50},
]

In [6]:
from maikol_utils.print_utils import print_separator
from dataclasses import replace

commets, bleus = [], []
for i, config in enumerate(configurations):
    print_separator(f"Configuration: {i}", sep_type="SUPER")

    CONFIG_AUX = replace(
        CONFIG,
        **config
    )
    # print(CONFIG_AUX)


    # ============= TRAIN ============
    tokenizer, lora_model = train_model(CONFIG_AUX)

    # ============= TEST ============
    print_separator("NORAMALIZE", sep_type="LONG")
    evaluate_model(CONFIG_AUX, tokenizer, lora_model)
    CONFIG_AUX = replace(
        CONFIG,
        normalize = False,
    )

    print_separator("NO NORAMALIZE", sep_type="LONG")
    evaluate_model(CONFIG_AUX, tokenizer, lora_model)

                                                        Configuration: 0                                                        

DatasetDict({
    train: Dataset({
        features: ['hypothesis', 'translation'],
        num_rows: 3339
    })
    test: Dataset({
        features: ['hypothesis', 'translation'],
        num_rows: 684
    })
})


Filter:   0%|          | 0/3339 [00:00<?, ? examples/s]

Filter:   0%|          | 0/684 [00:00<?, ? examples/s]

Map:   0%|          | 0/3337 [00:00<?, ? examples/s]

Map:   0%|          | 0/684 [00:00<?, ? examples/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


COMET: 77.32%
                                                        Configuration: 3                                                        

DatasetDict({
    train: Dataset({
        features: ['reference', 'translation'],
        num_rows: 3339
    })
    test: Dataset({
        features: ['reference', 'translation'],
        num_rows: 684
    })
})


Filter:   0%|          | 0/3339 [00:00<?, ? examples/s]

KeyError: 'hypothesis'